## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add current directory to path for local imports
sys.path.insert(0, os.getcwd())

from inference_ner import NERInference
from inference_classifier import AnimalClassifierInference, CLASS_NAMES
from main import run_pipeline, normalize_animal_name

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"\nSupported animal classes: {CLASS_NAMES}")

## 2. Exploratory Data Analysis

### 2.1 NER Training Data

The NER model was trained on synthetic sentences containing animal names. Below is an overview of the data generation process and label distribution.

In [ ]:
# Reproduce the synthetic NER dataset to analyze it
ANIMALS = [
    "butterfly", "cat", "chicken", "cow", "dog",
    "elephant", "horse", "sheep", "spider", "squirrel",
]

TEMPLATES = [
    "There is a {animal} in the picture.",
    "I can see a {animal} in this image.",
    "The {animal} is standing in the field.",
    "A {animal} was spotted near the river.",
    "Look at that beautiful {animal}!",
    "The photo shows a {animal} resting.",
    "Is that a {animal} over there?",
    "I think there is a {animal} in the background.",
    "A wild {animal} appeared in the garden.",
    "The {animal} looks very calm today.",
    "Can you see the {animal} on the left?",
    "That {animal} seems to be sleeping.",
    "We found a {animal} near the barn.",
    "The {animal} is eating grass.",
    "A small {animal} is hiding behind the tree.",
    "There are many {animal}s in this area.",
    "The {animal} jumped over the fence.",
    "I photographed a {animal} yesterday.",
    "This {animal} is quite large.",
    "Have you ever seen a {animal} this close?",
    "The little {animal} ran across the road.",
    "A friendly {animal} approached us.",
    "The {animal} was caught on camera.",
    "Someone left a picture of a {animal} here.",
    "My favorite animal is the {animal}.",
]

NEGATIVE_TEMPLATES = [
    "There is nothing in the picture.",
    "The weather is nice today.",
    "I can see a tree in this image.",
    "The field is empty and quiet.",
    "Look at that beautiful sunset!",
    "This photo shows a mountain landscape.",
    "Is that a car over there?",
    "The background is blurry.",
    "A wild flower appeared in the garden.",
    "Everything looks very calm today.",
]

# Generate dataset
random.seed(42)
num_samples = 500
positive_samples = []
negative_samples = []

for _ in range(int(num_samples * 0.8)):
    animal = random.choice(ANIMALS)
    template = random.choice(TEMPLATES)
    sentence = template.format(animal=animal)
    positive_samples.append({"sentence": sentence, "animal": animal, "type": "positive"})

for _ in range(int(num_samples * 0.2)):
    sentence = random.choice(NEGATIVE_TEMPLATES)
    negative_samples.append({"sentence": sentence, "animal": None, "type": "negative"})

all_samples = positive_samples + negative_samples
print(f"Total samples: {len(all_samples)}")
print(f"  Positive (with animal): {len(positive_samples)}")
print(f"  Negative (no animal):   {len(negative_samples)}")
print(f"\nSample positive: \"{positive_samples[0]['sentence']}\"")
print(f"Sample negative: \"{negative_samples[0]['sentence']}\"")

In [ ]:
# Distribution of animals in positive samples
from collections import Counter

animal_counts = Counter(s["animal"] for s in positive_samples)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: animal distribution
names, counts = zip(*sorted(animal_counts.items()))
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))
axes[0].bar(names, counts, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Animal Distribution in NER Training Data")
axes[0].set_xlabel("Animal Class")
axes[0].set_ylabel("Number of Samples")
axes[0].tick_params(axis="x", rotation=45)

# Pie chart: positive vs negative
axes[1].pie(
    [len(positive_samples), len(negative_samples)],
    labels=["Positive (animal)", "Negative (no animal)"],
    autopct="%1.1f%%",
    colors=["#66b3ff", "#ff9999"],
    startangle=90,
)
axes[1].set_title("Positive vs Negative Samples")

plt.tight_layout()
plt.show()

In [ ]:
# Token-level label distribution analysis
total_tokens = 0
animal_tokens = 0
o_tokens = 0

for s in positive_samples:
    tokens = s["sentence"].split()
    total_tokens += len(tokens)
    for tok in tokens:
        clean = tok.strip(".,!?;:'\"").lower()
        if clean == s["animal"] or clean == s["animal"] + "s":
            animal_tokens += 1
        else:
            o_tokens += 1

for s in negative_samples:
    tokens = s["sentence"].split()
    total_tokens += len(tokens)
    o_tokens += len(tokens)

print(f"Token-level statistics:")
print(f"  Total tokens:    {total_tokens}")
print(f"  B-ANIMAL tokens: {animal_tokens} ({animal_tokens/total_tokens*100:.1f}%)")
print(f"  O tokens:        {o_tokens} ({o_tokens/total_tokens*100:.1f}%)")
print(f"\nNote: Heavy class imbalance is normal for NER tasks.")
print(f"Most tokens are 'O' (outside any entity).")

### 2.2 Image Classification Model

The image classifier is a **ResNet-50** fine-tuned on the Animals-10 dataset (10 classes). Below we inspect the model architecture and parameters.

In [ ]:
# Load and inspect the image classification model
import torchvision.models as models

model = models.resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 10)

state_dict = torch.load("animals.pth", map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.eval()

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Image Classification Model: ResNet-50")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Number of classes:    10")
print(f"  Input size:           224x224 RGB")
print(f"\n  Classes: {CLASS_NAMES}")

# FC layer details
print(f"\n  Final FC layer: {model.fc}")
print(f"  FC weight shape: {model.fc.weight.shape}")

In [ ]:
# Inspect the NER model
from transformers import AutoModelForTokenClassification, AutoTokenizer

ner_model = AutoModelForTokenClassification.from_pretrained("ner-model")
ner_tokenizer = AutoTokenizer.from_pretrained("ner-model")

ner_params = sum(p.numel() for p in ner_model.parameters())

print("NER Model: BERT for Token Classification")
print(f"  Base model:       bert-base-cased")
print(f"  Total parameters: {ner_params:,}")
print(f"  Vocab size:       {ner_model.config.vocab_size:,}")
print(f"  Hidden size:      {ner_model.config.hidden_size}")
print(f"  Attention heads:  {ner_model.config.num_attention_heads}")
print(f"  Hidden layers:    {ner_model.config.num_hidden_layers}")
print(f"  Label mapping:    {ner_model.config.id2label}")

# Task 2: Named Entity Recognition + Image Classification Pipeline

This notebook demonstrates the full pipeline that:
1. Extracts animal names from text using a fine-tuned **BERT NER** model
2. Classifies animals in images using a fine-tuned **ResNet-50** model
3. Compares the results and returns a **boolean** verdict

## Table of Contents
1. Setup & Imports
2. Exploratory Data Analysis
3. NER Model Demo
4. Image Classifier Demo
5. Full Pipeline Demo
6. Edge Cases